## **Scopus**

Se necesita una key, los abstracts no se pueden recuperar directamente de scopus con cualquier key así que se necesita ver el abstract en otra página

In [ ]:
import requests
import pandas as pd
import time

# ── Configuración ──────────────────────────────────────────────
API_KEY  = Scopus_key
SCOPUS_URL  = 'https://api.elsevier.com/content/search/scopus'
OPENALEX_URL = 'https://api.openalex.org/works'

headers_scopus = {
    'X-ELS-APIKey': API_KEY,
    'Accept': 'application/json'
}

In [ ]:
def fetch_scopus(query, max_results=50):
    results = []
    start   = 0
    count   = 25

    while start < max_results:
        params = {
            'query': query,
            'count': count,
            'start': start
        }

        response = requests.get(SCOPUS_URL, headers=headers_scopus, params=params)

        if response.status_code != 200:
            print(f"Error Scopus: {response.status_code} - {response.text[:200]}")
            break

        entries = response.json().get('search-results', {}).get('entry', [])

        if not entries:
            print("Sin más resultados.")
            break

        for e in entries:
            results.append({
                'Title':       e.get('dc:title'),
                'Creator':     e.get('dc:creator'),
                'Publication': e.get('prism:publicationName'),
                'Date':        e.get('prism:coverDate'),
                'DOI':         e.get('prism:doi'),
                'Citations':   e.get('citedby-count')
            })

        print(f"  Scopus → {len(results)} artículos recuperados")
        start += count
        time.sleep(1)

    return pd.DataFrame(results)

In [ ]:
def reconstruct_abstract(inverted_index):
    if not inverted_index:
        return None

    positions = []
    for word, pos_list in inverted_index.items():
        for pos in pos_list:
            positions.append((pos, word))

    positions.sort(key=lambda x: x[0])
    return ' '.join(word for _, word in positions)


def fetch_abstract_openalex(doi):
    if not doi or pd.isna(doi):
        return "Sin DOI"

    params = {
        'filter': f'doi:{doi}',
        'select': 'abstract_inverted_index'
    }

    try:
        response = requests.get(OPENALEX_URL, params=params, timeout=10)

        if response.status_code != 200:
            return f"Error OpenAlex {response.status_code}"

        results = response.json().get('results', [])

        if not results:
            return "No encontrado en OpenAlex"

        abstract = reconstruct_abstract(results[0].get('abstract_inverted_index'))
        return abstract if abstract else "Sin abstract en OpenAlex"

    except requests.exceptions.Timeout:
        return "Timeout"
    except Exception as ex:
        return f"Error: {str(ex)}"


def add_abstracts(df):
    abstracts = []
    total = len(df)

    for i, doi in enumerate(df['DOI']):
        print(f"  OpenAlex abstract {i+1}/{total}  DOI: {doi}")
        abstracts.append(fetch_abstract_openalex(doi))
        time.sleep(0.3)   # OpenAlex pide no más de ~10 req/s

    df = df.copy()
    df['Abstract'] = abstracts
    return df


In [40]:
query = 'TITLE ( ( "TUBERCULOSIS" ) AND ( "IDENTIF*" OR "DISCOVERY" OR "DETERMINATION" ) AND ( "TARGET*" ) AND NOT ( "IN SILICO" OR "BOVIS" OR "BOVINE" ) ) AND PUBYEAR > 2009 AND PUBYEAR < 2027 AND ( DOCTYPE ( "ar" ) OR DOCTYPE ( "re" ) OR DOCTYPE ( "cp" ) ) AND ABS ( "IN VITRO" OR "IN VIVO" OR "VALIDAT*" OR "EVALUAT*" OR "EXPERIMENTAL" OR "DRUG TARGETS" ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )'
df_scopus = fetch_scopus(query, max_results=200)
df_final = add_abstracts(df_scopus)
df_final.to_csv('targets_scopus.csv', index=False, encoding='utf-8-sig')

  Scopus → 25 artículos recuperados
  Scopus → 50 artículos recuperados
  Scopus → 75 artículos recuperados
  Scopus → 100 artículos recuperados
  Scopus → 119 artículos recuperados
Sin más resultados.
  OpenAlex abstract 1/119  DOI: 10.1038/s41598-026-46983-z
  OpenAlex abstract 2/119  DOI: 10.1142/S2737416525501121
  OpenAlex abstract 3/119  DOI: 10.1007/s00203-026-05005-2
  OpenAlex abstract 4/119  DOI: 10.1016/j.compbiomed.2026.111808
  OpenAlex abstract 5/119  DOI: 10.1016/j.bmc.2026.118572
  OpenAlex abstract 6/119  DOI: 10.1007/s10529-026-03690-z
  OpenAlex abstract 7/119  DOI: 10.1016/j.jmgm.2025.109220
  OpenAlex abstract 8/119  DOI: 10.2147/IDR.S571825
  OpenAlex abstract 9/119  DOI: 10.1007/s10822-025-00695-0
  OpenAlex abstract 10/119  DOI: 10.1016/j.micpath.2025.107949
  OpenAlex abstract 11/119  DOI: 10.1016/j.rechem.2025.102388
  OpenAlex abstract 12/119  DOI: 10.1016/j.jmgm.2025.108954
  OpenAlex abstract 13/119  DOI: 10.1021/acsinfecdis.4c00808
  OpenAlex abstract 14/1

## **PubMed**

In [ ]:
import requests
import pandas as pd
import time

# ── Configuración ──────────────────────────────────────────────
API_KEY     = PubMed_key
PUBMED_SEARCH_URL  = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi'
PUBMED_FETCH_URL   = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi'
PUBMED_SUMMARY_URL = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi'
OPENALEX_URL       = 'https://api.openalex.org/works'

In [ ]:
def search_pubmed_ids(query, max_results=50):
    params = {
        'db':      'pubmed',
        'term':    query,
        'retmax':  max_results,
        'retmode': 'json',
        'api_key': API_KEY
    }

    response = requests.get(PUBMED_SEARCH_URL, params=params)

    if response.status_code != 200:
        print(f"Error PubMed search: {response.status_code} - {response.text[:200]}")
        return []

    data    = response.json()
    id_list = data.get('esearchresult', {}).get('idlist', [])
    print(f"  PubMed → {len(id_list)} PMIDs encontrados")
    return id_list


def fetch_pubmed_metadata(pmid_list):
    if not pmid_list:
        return pd.DataFrame()

    chunk_size = 100
    all_results = []

    for i in range(0, len(pmid_list), chunk_size):
        chunk = pmid_list[i:i + chunk_size]
        ids_str = ','.join(chunk)

        params = {
            'db':      'pubmed',
            'id':      ids_str,
            'retmode': 'json',
            'api_key': API_KEY
        }

        response = requests.get(PUBMED_SUMMARY_URL, params=params)

        if response.status_code != 200:
            print(f"Error esummary: {response.status_code}")
            continue

        data    = response.json()
        records = data.get('result', {})

        for pmid in chunk:
            record = records.get(pmid, {})
            authors     = record.get('authors', [])
            first_author = authors[0].get('name', 'Unknown') if authors else 'Unknown'

            doi = None
            for id_obj in record.get('articleids', []):
                if id_obj.get('idtype') == 'doi':
                    doi = id_obj.get('value')
                    break

            all_results.append({
                'PMID':        pmid,
                'Title':       record.get('title', '').rstrip('.'),
                'Creator':     first_author,
                'Publication': record.get('fulljournalname', record.get('source', '')),
                'Date':        record.get('pubdate', ''),
                'DOI':         doi,
                'Citations':   None   
            })

        print(f"  Metadatos recuperados: {len(all_results)}")
        time.sleep(0.4)   
    return pd.DataFrame(all_results)


def reconstruct_abstract(inverted_index):
    if not inverted_index:
        return None

    positions = []
    for word, pos_list in inverted_index.items():
        for pos in pos_list:
            positions.append((pos, word))

    positions.sort(key=lambda x: x[0])
    return ' '.join(word for _, word in positions)


def fetch_abstract_openalex(doi=None, pmid=None):
    filtro = None

    if doi and not pd.isna(doi):
        filtro = f'doi:{doi}'
    elif pmid:
        filtro = f'ids.pmid:{pmid}'
    else:
        return "Sin DOI ni PMID"

    params = {
        'filter': filtro,
        'select': 'abstract_inverted_index'
    }

    try:
        response = requests.get(OPENALEX_URL, params=params, timeout=10)

        if response.status_code != 200:
            return f"Error OpenAlex {response.status_code}"

        results = response.json().get('results', [])

        if not results:
            return "No encontrado en OpenAlex"

        abstract = reconstruct_abstract(results[0].get('abstract_inverted_index'))
        return abstract if abstract else "Sin abstract en OpenAlex"

    except requests.exceptions.Timeout:
        return "Timeout"
    except Exception as ex:
        return f"Error: {str(ex)}"


def add_abstracts(df):
    abstracts = []
    total = len(df)

    for i, row in df.iterrows():
        print(f"  OpenAlex abstract {i+1}/{total}  PMID: {row['PMID']}  DOI: {row['DOI']}")

        abstract = fetch_abstract_openalex(
            doi=row.get('DOI'),
            pmid=row.get('PMID')
        )
        abstracts.append(abstract)
        time.sleep(0.3)

    df = df.copy()
    df['Abstract'] = abstracts
    return df


PASO 1 — Búsqueda de PMIDs en PubMed
  PubMed → 2 PMIDs encontrados

PASO 2 — Metadatos desde PubMed
  Metadatos recuperados: 2

Total artículos: 2
Con DOI:         2
Sin DOI:         0 (se buscarán por PMID)

PASO 3 — Abstracts desde OpenAlex (join por DOI o PMID)
  OpenAlex abstract 1/2  PMID: 42366204  DOI: 10.1038/s41598-026-59621-5
  OpenAlex abstract 2/2  PMID: 42362603  DOI: 10.1038/s41597-026-07689-z

Abstracts recuperados: 1 / 2

Archivo guardado → pubmed_openalex.csv
                                                                                                                                            Title                                                                                                                                                                                                                                                                                                                                                                                     

In [ ]:
query = 'machine learning AND renewable energy'
pmid_list = search_pubmed_ids(query, max_results=2)
df_pubmed = fetch_pubmed_metadata(pmid_list)
df_final = add_abstracts(df_pubmed)

df_final.to_csv('targets_pubmed.csv', index=False, encoding='utf-8-sig')